# Benin Insights 2026
## Pipeline GDELT - Visualisation interactive
**iSHEERO x DataCamp Donates - Role : Data Engineer**

> Lance toutes les cellules dans l'ordre : Runtime -> Run all

---

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'pandas', 'numpy'], check=True)
print('Packages prets')

## 1 - Generation des donnees prototype GDELT Benin

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import json, warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
N = 8000

event_codes = {
    '010': 'Declaration publique',
    '020': 'Appel action',
    '036': 'Critique politique',
    '040': 'Consultation diplomatique',
    '050': 'Appel cooperation',
    '060': 'Cooperation materielle',
    '070': 'Aide/assistance',
    '100': 'Demande',
    '110': 'Rejet/opposition',
    '120': 'Menace',
    '130': 'Protestation',
    '140': 'Refus cooperation',
    '145': 'Manifestation/revolte',
    '170': 'Coercition',
    '180': 'Attaque',
}
ek = list(event_codes.keys())
ew = np.array([.12,.08,.10,.09,.08,.07,.06,.07,.07,.04,.06,.04,.03,.03,.03])
ew /= ew.sum()
chosen_ev = np.random.choice(ek, N, p=ew)

cities_info = {
    'Cotonou':       {'lat':6.3654,  'lon':2.4183,  'zone':'Sud',    'dept':'Littoral',   'nord':False},
    'Porto-Novo':    {'lat':6.4969,  'lon':2.6289,  'zone':'Sud',    'dept':'Oueme',      'nord':False},
    'Parakou':       {'lat':9.3370,  'lon':2.6282,  'zone':'Nord',   'dept':'Borgou',     'nord':True},
    'Abomey-Calavi': {'lat':6.4484,  'lon':2.3552,  'zone':'Sud',    'dept':'Atlantique', 'nord':False},
    'Bohicon':       {'lat':7.1824,  'lon':2.0660,  'zone':'Centre', 'dept':'Zou',        'nord':False},
    'Natitingou':    {'lat':10.3081, 'lon':1.3784,  'zone':'Nord',   'dept':'Atacora',    'nord':True},
    'Kandi':         {'lat':11.1342, 'lon':2.9394,  'zone':'Nord',   'dept':'Alibori',    'nord':True},
    'Lokossa':       {'lat':6.6392,  'lon':1.7153,  'zone':'Sud',    'dept':'Mono',       'nord':False},
    'Ouidah':        {'lat':6.3600,  'lon':2.0852,  'zone':'Sud',    'dept':'Atlantique', 'nord':False},
    'Abomey':        {'lat':7.1856,  'lon':1.9861,  'zone':'Centre', 'dept':'Zou',        'nord':False},
    'Djougou':       {'lat':9.7082,  'lon':1.6599,  'zone':'Nord',   'dept':'Donga',      'nord':True},
    'Malanville':    {'lat':11.8683, 'lon':3.3836,  'zone':'Nord',   'dept':'Alibori',    'nord':True},
}
city_names = list(cities_info.keys())
city_weights = np.array([.303,.151,.124,.100,.060,.050,.040,.040,.040,.035,.030,.027])
city_weights /= city_weights.sum()
chosen_cities = np.random.choice(city_names, N, p=city_weights)

actors = ['Gouvernement','Parti politique','ONG','Militaire','Medias',
          'Diplomate','Syndicat','Citoyen','Police','Institution internationale']
actor_w = np.array([.25,.18,.12,.08,.12,.07,.06,.05,.04,.03])
actor_w /= actor_w.sum()

sources = ['bbc.com','rfi.fr','beninwebtv.com','lanouvelletribune.bj',
           'matinlibre.com','24haubenin.com','fraternite.bj','apanews.net']
source_regions = {'bbc.com':'International','rfi.fr':'International',
                  'beninwebtv.com':'Benin','lanouvelletribune.bj':'Benin',
                  'matinlibre.com':'Benin','24haubenin.com':'Benin',
                  'fraternite.bj':'Benin','apanews.net':'Afrique'}

start_date = datetime(2025, 1, 1)
end_date   = datetime(2025, 12, 31)
date_range = (end_date - start_date).days + 1
dates = [start_date + timedelta(days=int(d)) for d in np.random.choice(date_range, N)]

neg_events = ['110','120','130','140','145','170','180']
is_neg = np.isin(chosen_ev, neg_events)

goldstein = np.where(is_neg,
    np.random.normal(-3.5, 2.5, N),
    np.random.normal(2.0, 2.0, N)).clip(-10, 10)

avg_tone = np.where(is_neg,
    np.random.normal(-4.2, 3.0, N),
    np.random.normal(1.8, 2.5, N)).clip(-20, 20)

quad_class = np.where(chosen_ev == '060', 2,
             np.where(chosen_ev == '070', 2,
             np.where(np.isin(chosen_ev, ['145','170','180']), 4,
             np.where(is_neg, 3, 1))))

num_mentions = np.random.negative_binomial(3, 0.3, N) + 1
num_articles = np.random.negative_binomial(2, 0.4, N) + 1
chosen_sources = np.random.choice(sources, N)
chosen_actors  = np.random.choice(actors, N, p=actor_w)

df = pd.DataFrame({
    'SQLDATE':            [d.strftime('%Y%m%d') for d in dates],
    'date':               dates,
    'Actor1Name':         chosen_actors,
    'Actor1CountryCode':  np.where(np.random.rand(N)<0.7,'BN',
                            np.random.choice(['FR','US','NG','GH','TG'],N)),
    'EventCode':          chosen_ev,
    'EventLabel':         [event_codes[e] for e in chosen_ev],
    'QuadClass':          quad_class,
    'GoldsteinScale':     goldstein.round(2),
    'NumMentions':        num_mentions,
    'NumArticles':        num_articles,
    'AvgTone':            avg_tone.round(2),
    'ActionGeo_FullName': chosen_cities,
    'ActionGeo_Lat':      [cities_info[c]['lat'] for c in chosen_cities],
    'ActionGeo_Long':     [cities_info[c]['lon'] for c in chosen_cities],
    'SourceDomain':       chosen_sources,
})

# ── GKG (Global Knowledge Graph) columns ──────────────────────────────
gkg_themes_pool = [
    'POLITICAL_VIOLENCE','ELECTIONS','DEMOCRACY','HUMAN_RIGHTS',
    'ECONOMY','HEALTH','EDUCATION','RELIGION','MILITARY','ENVIRONMENT',
    'TERRORISM','PROTEST','DIPLOMACY','CORRUPTION','DEVELOPMENT'
]
gkg_persons_pool = [
    'Patrice Talon','Alassane Ouattara','Macky Sall','Paul Biya',
    'Mahamadou Issoufou','Thomas Boni Yayi','Lionel Zinsou','Abdoulaye Bio Tchane'
]
gkg_orgs_pool = [
    'CEDEAO','Union Africaine','ONU','Banque Mondiale','FMI',
    'OMS','UNICEF','PNUD','Gouvernement Beninois','Assemblee Nationale'
]

def random_semicolon_list(pool, k_max=3):
    k = np.random.randint(0, k_max + 1)
    if k == 0:
        return ''
    return ';'.join(np.random.choice(pool, k, replace=False).tolist())

np.random.seed(43)  # reproductibilite GKG
df['GKG_Themes']       = [random_semicolon_list(gkg_themes_pool, 4) for _ in range(N)]
df['GKG_Persons']      = [random_semicolon_list(gkg_persons_pool, 2) for _ in range(N)]
df['GKG_Organizations']= [random_semicolon_list(gkg_orgs_pool, 3) for _ in range(N)]
df['GKG_Locations']    = df['ActionGeo_FullName'] + ', Benin'
# V2Tone: 6 composantes separees par virgule (tone,pos,neg,polarity,arXiv,wordcount)
v2tone_tone = avg_tone.round(2)
v2tone_pos  = np.abs(np.where(avg_tone > 0, avg_tone, 0)).round(2)
v2tone_neg  = np.abs(np.where(avg_tone < 0, avg_tone, 0)).round(2)
v2tone_pol  = (v2tone_pos + v2tone_neg).round(2)
v2tone_arx  = np.random.uniform(0, 2, N).round(2)
v2tone_wc   = np.random.randint(50, 800, N)
df['GKG_V2Tone']       = [f'{t},{p},{n},{pol},{a},{w}'
                          for t,p,n,pol,a,w in
                          zip(v2tone_tone,v2tone_pos,v2tone_neg,
                              v2tone_pol,v2tone_arx,v2tone_wc)]
df['GKG_SharingImage'] = np.random.choice(
    ['https://www.bbc.com/img/default.jpg',
     'https://www.rfi.fr/img/placeholder.jpg',
     ''], N, p=[0.15, 0.15, 0.70])
df['GKG_DocId']        = [
    f"{d.strftime('%Y%m%d%H%M%S')}-{src.replace('.','_')}-BN-{idx:05d}"
    for idx,(d,src) in enumerate(zip(dates, df['SourceDomain'].tolist()))
]
# ── Fin GKG ─────────────────────────────────────────────────────────────

quad_labels = {1:'Cooperation verbale',2:'Cooperation materielle',
               3:'Conflit verbal',4:'Conflit materiel'}
df['QuadLabel']       = df['QuadClass'].map(quad_labels)
df['IsConflict']      = (df['QuadClass'] >= 3).astype(int)
df['IsNegative']      = (df['AvgTone'] < 0).astype(int)
df['MediaWeight']     = df['NumMentions'] * df['NumArticles']
df['IsNorthBenin']    = df['ActionGeo_FullName'].map(lambda c: int(cities_info[c]['nord']))
df['ZoneBenin']       = df['ActionGeo_FullName'].map(lambda c: cities_info[c]['zone'])
df['DepartementBenin']= df['ActionGeo_FullName'].map(lambda c: cities_info[c]['dept'])
df['SourceRegion']    = df['SourceDomain'].map(source_regions)
df['GoldsteinNorm']   = ((df['GoldsteinScale'] + 10) / 20).round(4)
df['ToneNorm']        = ((df['AvgTone'] + 20) / 40).round(4)
df['month']           = pd.to_datetime(df['date']).dt.to_period('M')

def tone_cat(t):
    if t < -5: return 'Tres negatif'
    elif t < -2: return 'Negatif'
    elif t < 2: return 'Neutre'
    elif t < 5: return 'Positif'
    else: return 'Tres positif'

def gold_bin(g):
    if g < -7: return 'Tres destabilisant'
    elif g < -3: return 'Destabilisant'
    elif g < 0: return 'Leg. negatif'
    elif g < 3: return 'Neutre'
    elif g < 7: return 'Stabilisant'
    else: return 'Tres stabilisant'

df['ToneCategory'] = df['AvgTone'].apply(tone_cat)
df['GoldsteinBin'] = df['GoldsteinScale'].apply(gold_bin)

print(f'Dataset genere : {len(df):,} evenements, {df.shape[1]} colonnes')
print(f'Periode : {df["date"].min().date()} -> {df["date"].max().date()}')
print(f'Taux conflit : {df["IsConflict"].mean()*100:.1f}%')
print(f'AvgTone moy  : {df["AvgTone"].mean():.3f}')
print(f'Goldstein moy: {df["GoldsteinScale"].mean():.3f}')
df.head(3)

## 2 - Rapport pipeline

In [ ]:
print('='*50)
print('  RAPPORT PIPELINE GDELT BENIN (BN)')
print('='*50)
print(f'\nVOLUME')
print(f'  Total     : {len(df):,}')
print(f'  Features  : {df.shape[1]}')
print(f'  Periode   : {df["date"].min().date()} -> {df["date"].max().date()}')
print(f'\nINDICATEURS')
print(f'  AvgTone   : {df["AvgTone"].mean():.3f}')
print(f'  Goldstein : {df["GoldsteinScale"].mean():.3f}')
print(f'  Conflit   : {df["IsConflict"].mean()*100:.1f}%')
print(f'  Negatif   : {df["IsNegative"].mean()*100:.1f}%')
print(f'\nTOP 5 VILLES')
for city, cnt in df['ActionGeo_FullName'].value_counts().head(5).items():
    print(f'  {city:<20} {cnt:>5}')
print(f'\nQUADCLASS')
for lbl, cnt in df['QuadLabel'].value_counts().items():
    print(f'  {lbl:<30} {cnt:>5} ({cnt/len(df)*100:.1f}%)')
print('='*50)

## 3 - Dashboard interactif dans Colab

In [ ]:
from IPython.display import HTML, display

monthly = df.groupby('month').size().reset_index(name='count').tail(12)
months_labels = [str(m) for m in monthly['month'].tolist()]
months_vals   = monthly['count'].tolist()

tone_by_month = df.groupby('month')['AvgTone'].mean().reset_index().tail(12)
gold_by_month = df.groupby('month')['GoldsteinScale'].mean().reset_index().tail(12)
tone_vals = [round(float(v), 2) for v in tone_by_month['AvgTone'].tolist()]
gold_vals = [round(float(v), 2) for v in gold_by_month['GoldsteinScale'].tolist()]

conflict_by_month = df.groupby('month')['IsConflict'].mean().reset_index().tail(12)
conflict_vals = [round(float(v)*100, 1) for v in conflict_by_month['IsConflict'].tolist()]

quad_counts = df['QuadLabel'].value_counts()
quad_labels_list = quad_counts.index.tolist()
quad_vals_list   = [int(v) for v in quad_counts.values.tolist()]

top_cities = df['ActionGeo_FullName'].value_counts().head(8)
city_labels = top_cities.index.tolist()
city_counts = [int(v) for v in top_cities.values.tolist()]

tone_dist = df['ToneCategory'].value_counts()
tone_cats = tone_dist.index.tolist()
tone_cnts = [int(v) for v in tone_dist.values.tolist()]

sample = df[['date','Actor1Name','EventLabel','ActionGeo_FullName',
             'GoldsteinScale','AvgTone','ToneCategory','QuadLabel']].head(10)
sample_rows = []
for _, r in sample.iterrows():
    sample_rows.append({
        'date': str(r['date'])[:10],
        'actor': str(r['Actor1Name']),
        'event': str(r['EventLabel']),
        'city': str(r['ActionGeo_FullName']),
        'goldstein': round(float(r['GoldsteinScale']), 2),
        'tone': round(float(r['AvgTone']), 2),
        'sentiment': str(r['ToneCategory']),
        'quad': str(r['QuadLabel'])
    })

total_events  = len(df)
n_features    = int(df.shape[1])
avg_tone_val  = round(float(df['AvgTone'].mean()), 2)
conflict_pct  = round(float(df['IsConflict'].mean() * 100), 1)
goldstein_val = round(float(df['GoldsteinScale'].mean()), 2)
n_cities      = int(df['ActionGeo_FullName'].nunique())
period_start  = str(df['date'].min().date())
period_end    = str(df['date'].max().date())

tone_color    = '#D85A30' if avg_tone_val < 0 else '#1D9E75'
gold_color    = '#D85A30' if goldstein_val < 0 else '#1D9E75'
tone_sign     = '+' if avg_tone_val >= 0 else ''
gold_sign     = '+' if goldstein_val >= 0 else ''

print('Donnees dashboard preparees')
print(f'  {len(months_labels)} mois | {len(city_labels)} villes | {len(sample_rows)} lignes tableau')

In [ ]:
HTML_TEMPLATE = '''
<!DOCTYPE html><html lang="fr"><head><meta charset="UTF-8">
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.js"></script>
<style>
*{box-sizing:border-box;margin:0;padding:0;font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif}
body{background:#f8f7f4;color:#1a1a18;padding:16px;font-size:14px}
.header{background:#fff;border-radius:12px;padding:18px 22px;margin-bottom:14px;border:1px solid #e8e6de}
.h-title{font-size:19px;font-weight:600;margin-bottom:4px}
.h-sub{font-size:12px;color:#888;margin-bottom:10px}
.badge{display:inline-block;font-size:11px;padding:3px 10px;border-radius:20px;background:#E1F5EE;color:#0F6E56;font-weight:500;margin-right:6px;margin-bottom:4px}
.pipeline{display:flex;align-items:center;gap:5px;flex-wrap:wrap;margin-top:10px}
.ps{font-size:11px;padding:3px 10px;border-radius:14px;border:1px solid #e0ddd5;color:#5F5E5A;background:#fff}
.pa{font-size:11px;color:#B4B2A9}
.status{display:flex;align-items:center;gap:8px;font-size:12px;color:#5F5E5A;background:#f0f0ec;border-radius:8px;padding:8px 14px;margin-bottom:14px}
.sdot{width:8px;height:8px;border-radius:50%;background:#1D9E75;flex-shrink:0}
.metrics{display:grid;grid-template-columns:repeat(3,1fr);gap:10px;margin-bottom:14px}
.metric{background:#fff;border-radius:10px;padding:14px 16px;border:1px solid #e8e6de}
.ml{font-size:11px;color:#888;margin-bottom:5px;font-weight:500;text-transform:uppercase;letter-spacing:.4px}
.mv{font-size:22px;font-weight:600}
.ms{font-size:11px;color:#B4B2A9;margin-top:2px}
.grid2{display:grid;grid-template-columns:1fr 1fr;gap:12px;margin-bottom:12px}
.card{background:#fff;border-radius:10px;padding:16px;border:1px solid #e8e6de}
.card.full{grid-column:1/-1}
.ct{font-size:13px;font-weight:600;margin-bottom:2px}
.cs{font-size:11px;color:#888;margin-bottom:12px}
.legend{display:flex;flex-wrap:wrap;gap:10px;margin-bottom:10px;font-size:11px;color:#5F5E5A}
.li{display:flex;align-items:center;gap:4px}
.ld{width:10px;height:10px;border-radius:2px;display:inline-block;flex-shrink:0}
.bar-row{display:flex;align-items:center;gap:8px;margin-bottom:7px}
.bl{font-size:11px;color:#5F5E5A;width:100px;text-align:right;flex-shrink:0}
.bt{flex:1;height:16px;background:#f0f0ec;border-radius:4px;overflow:hidden}
.bf{height:100%;border-radius:4px}
.bv{font-size:11px;color:#888;width:50px;flex-shrink:0}
table{width:100%;border-collapse:collapse;font-size:12px}
th{text-align:left;color:#888;font-weight:500;padding:7px 8px;border-bottom:1px solid #f0f0ec;font-size:11px}
td{padding:7px 8px;border-bottom:1px solid #f8f7f4;color:#1a1a18}
tr:last-child td{border-bottom:none}
tr:hover td{background:#fafaf8}
.pill{display:inline-block;font-size:10px;padding:2px 8px;border-radius:12px;font-weight:500}
.pc{background:#E1F5EE;color:#0F6E56}
.pf{background:#FAECE7;color:#993C1D}
.pn{background:#FCEBEB;color:#A32D2D}
.pp{background:#EAF3DE;color:#3B6D11}
.pu{background:#F1EFE8;color:#5F5E5A}
</style></head><body>
CONTENT_PLACEHOLDER
</body></html>
'''

content = f'''
<div class="header">
  <div class="h-title">Benin Insights 2026 - Dashboard Pipeline</div>
  <div class="h-sub">BigQuery GDELT v2 - Data Engineer - iSHEERO x DataCamp Donates</div>
  <div>
    <span class="badge">GDELT v2</span>
    <span class="badge">Benin (BN)</span>
    <span class="badge">Prototype local</span>
  </div>
  <div class="pipeline">
    <span class="ps">BigQuery GDELT</span><span class="pa">-&gt;</span>
    <span class="ps">Extraction mensuelle</span><span class="pa">-&gt;</span>
    <span class="ps">Nettoyage</span><span class="pa">-&gt;</span>
    <span class="ps">{n_features} features</span><span class="pa">-&gt;</span>
    <span class="ps">gdelt_benin_clean.csv</span>
  </div>
</div>

<div class="status">
  <span class="sdot"></span>
  <span>Mode prototype - {total_events:,} evenements - {n_cities} villes - {n_features} features - {period_start} a {period_end}</span>
</div>

<div class="metrics">
  <div class="metric"><div class="ml">Evenements total</div><div class="mv">{total_events:,}</div><div class="ms">12 mois Benin</div></div>
  <div class="metric"><div class="ml">Features</div><div class="mv">{n_features}</div><div class="ms">variables calculees</div></div>
  <div class="metric"><div class="ml">AvgTone moyen</div><div class="mv" style="color:{tone_color}">{tone_sign}{avg_tone_val}</div><div class="ms">sentiment presse</div></div>
  <div class="metric"><div class="ml">Taux conflit</div><div class="mv" style="color:#D85A30">{conflict_pct}%</div><div class="ms">QuadClass &gt;= 3</div></div>
  <div class="metric"><div class="ml">Goldstein moyen</div><div class="mv" style="color:{gold_color}">{gold_sign}{goldstein_val}</div><div class="ms">stabilite politique</div></div>
  <div class="metric"><div class="ml">Villes</div><div class="mv">{n_cities}</div><div class="ms">localites du Benin</div></div>
</div>

<div class="grid2">
  <div class="card">
    <div class="ct">Evenements par mois</div>
    <div class="cs">Volume mensuel GDELT - 12 mois</div>
    <div style="position:relative;height:200px">
      <canvas id="cMois" role="img" aria-label="Histogramme mensuel">Donnees mensuelles.</canvas>
    </div>
  </div>
  <div class="card">
    <div class="ct">QuadClass - Repartition</div>
    <div class="cs">Type evenements CAMEO</div>
    <div class="legend" id="qLeg"></div>
    <div style="position:relative;height:165px">
      <canvas id="cQuad" role="img" aria-label="Repartition QuadClass">4 types evenements.</canvas>
    </div>
  </div>
  <div class="card">
    <div class="ct">AvgTone et GoldsteinScale</div>
    <div class="cs">Tonalite et stabilite par mois</div>
    <div class="legend">
      <span class="li"><span class="ld" style="background:#1D9E75"></span>AvgTone</span>
      <span class="li"><span class="ld" style="background:#D85A30"></span>Goldstein</span>
    </div>
    <div style="position:relative;height:180px">
      <canvas id="cTone" role="img" aria-label="Courbes AvgTone et Goldstein">Indicateurs mensuels.</canvas>
    </div>
  </div>
  <div class="card">
    <div class="ct">Taux de conflit mensuel</div>
    <div class="cs">Pourcentage QuadClass &gt;= 3</div>
    <div style="position:relative;height:200px">
      <canvas id="cConflict" role="img" aria-label="Taux conflit mensuel">Evolution conflit.</canvas>
    </div>
  </div>
  <div class="card">
    <div class="ct">Top 8 villes</div>
    <div class="cs">Volume par localite</div>
    <div id="barC"></div>
  </div>
  <div class="card">
    <div class="ct">Sentiment - Distribution</div>
    <div class="cs">Categories ToneCategory</div>
    <div style="position:relative;height:230px">
      <canvas id="cSent" role="img" aria-label="Distribution sentiment">5 categories.</canvas>
    </div>
  </div>
  <div class="card full">
    <div class="ct" style="margin-bottom:4px">Echantillon dataset - gdelt_benin_clean.csv</div>
    <div class="cs">10 premiers evenements avec feature engineering</div>
    <div style="overflow-x:auto">
      <table><thead><tr>
        <th>Date</th><th>Acteur</th><th>Evenement</th><th>Ville</th>
        <th>Goldstein</th><th>AvgTone</th><th>Sentiment</th><th>QuadClass</th>
      </tr></thead><tbody id="tBody"></tbody></table>
    </div>
  </div>
</div>

<script>
var months   = ''' + json.dumps(months_labels) + ''';
var evD      = ''' + json.dumps(months_vals) + ''';
var toneD    = ''' + json.dumps(tone_vals) + ''';
var goldD    = ''' + json.dumps(gold_vals) + ''';
var conflD   = ''' + json.dumps(conflict_vals) + ''';
var qLabels  = ''' + json.dumps(quad_labels_list) + ''';
var qVals    = ''' + json.dumps(quad_vals_list) + ''';
var cLabels  = ''' + json.dumps(city_labels) + ''';
var cVals    = ''' + json.dumps(city_counts) + ''';
var sLabels  = ''' + json.dumps(tone_cats) + ''';
var sVals    = ''' + json.dumps(tone_cnts) + ''';
var rows     = ''' + json.dumps(sample_rows) + ''';

var QC = ["#1D9E75","#378ADD","#EF9F27","#D85A30"];
var SC = ["#E24B4A","#D85A30","#888780","#1D9E75","#3B6D11"];

new Chart(document.getElementById("cMois"),{
  type:"bar",
  data:{labels:months,datasets:[{label:"Evenements",data:evD,backgroundColor:"rgba(29,158,117,0.75)",borderRadius:4}]},
  options:{responsive:true,maintainAspectRatio:false,plugins:{legend:{display:false}},
    scales:{x:{ticks:{font:{size:10},autoSkip:false,maxRotation:45}},y:{ticks:{font:{size:10}}}}}
});

var ql=document.getElementById("qLeg");
qLabels.forEach(function(l,i){
  var pct=Math.round(qVals[i]/qVals.reduce(function(a,b){return a+b;},0)*100);
  ql.innerHTML+=\'<span class="li"><span class="ld" style="background:\'+QC[i]+\'"></span>\'+l+\' \'+pct+\'%</span>\';
});
new Chart(document.getElementById("cQuad"),{
  type:"doughnut",
  data:{labels:qLabels,datasets:[{data:qVals,backgroundColor:QC,borderWidth:0}]},
  options:{responsive:true,maintainAspectRatio:false,cutout:"60%",plugins:{legend:{display:false}}}
});

new Chart(document.getElementById("cTone"),{
  type:"line",
  data:{labels:months,datasets:[
    {label:"AvgTone",data:toneD,borderColor:"#1D9E75",backgroundColor:"rgba(29,158,117,0.08)",tension:0.4,pointRadius:3,fill:true},
    {label:"Goldstein",data:goldD,borderColor:"#D85A30",backgroundColor:"rgba(216,90,48,0.05)",tension:0.4,pointRadius:3,borderDash:[4,3]}
  ]},
  options:{responsive:true,maintainAspectRatio:false,plugins:{legend:{display:false}},
    scales:{x:{ticks:{font:{size:10},autoSkip:false,maxRotation:45}},y:{ticks:{font:{size:10}}}}}
});

new Chart(document.getElementById("cConflict"),{
  type:"line",
  data:{labels:months,datasets:[{label:"Conflit %",data:conflD,borderColor:"#D85A30",backgroundColor:"rgba(216,90,48,0.1)",tension:0.4,fill:true,pointRadius:4}]},
  options:{responsive:true,maintainAspectRatio:false,plugins:{legend:{display:false}},
    scales:{x:{ticks:{font:{size:10},autoSkip:false,maxRotation:45}},y:{min:0,max:60,ticks:{font:{size:10},callback:function(v){return v+"%";}}}}}
});

var maxC=Math.max.apply(null,cVals);
var bc=document.getElementById("barC");
var bColors=["#1D9E75","#1D9E75","#378ADD","#378ADD","#888780","#888780","#888780","#888780"];
bc.innerHTML=cLabels.map(function(c,i){
  return \'<div class="bar-row"><div class="bl">\'+c+\'</div><div class="bt"><div class="bf" style="width:\'+Math.round(cVals[i]/maxC*100)+\'%;background:\'+bColors[i]+\'"></div></div><div class="bv">\'+cVals[i].toLocaleString()+\'</div></div>\';
}).join("");

new Chart(document.getElementById("cSent"),{
  type:"bar",
  data:{labels:sLabels,datasets:[{label:"Evenements",data:sVals,backgroundColor:SC,borderRadius:4}]},
  options:{responsive:true,maintainAspectRatio:false,indexAxis:"y",plugins:{legend:{display:false}},
    scales:{x:{ticks:{font:{size:10}}},y:{ticks:{font:{size:10}}}}}
});

document.getElementById("tBody").innerHTML=rows.map(function(r){
  var gc=r.goldstein>=0?"#1D9E75":"#D85A30";
  var tc=r.tone>=0?"#1D9E75":"#D85A30";
  var qc=r.quad.indexOf("Conflit")>=0?"pf":"pc";
  var sc=(r.sentiment.indexOf("negatif")>=0||r.sentiment.indexOf("Negatif")>=0)?"pn":(r.sentiment.indexOf("Positif")>=0?"pp":"pu");
  var gs=(r.goldstein>=0?"+":"")+r.goldstein.toFixed(1);
  var ts=(r.tone>=0?"+":"")+r.tone.toFixed(1);
  return "<tr><td>"+r.date+"</td><td>"+r.actor+"</td><td>"+r.event+"</td><td>"+r.city+"</td><td style=\\"font-weight:600;color:"+gc+"\\">"+gs+"</td><td style=\\"font-weight:600;color:"+tc+"\\">"+ts+"</td><td><span class=\\"pill "+sc+"\\">"+r.sentiment+"</span></td><td><span class=\\"pill "+qc+"\\">"+r.quad+"</span></td></tr>";
}).join("");
</script>
'''

dashboard_html = HTML_TEMPLATE.replace('CONTENT_PLACEHOLDER', content)
display(HTML(dashboard_html))
print('Dashboard affiche dans Colab')

## 4 - Sauvegarder et telecharger les fichiers

In [ ]:
import os, json
from pathlib import Path

os.makedirs('data/processed', exist_ok=True)
os.makedirs('dashboard', exist_ok=True)

csv_path = 'data/processed/gdelt_benin_clean.csv'
df.to_csv(csv_path, index=False, encoding='utf-8')
print(f'CSV : {csv_path} ({Path(csv_path).stat().st_size/1e6:.1f} MB)')

html_path = 'dashboard/index.html'
with open(html_path, 'w', encoding='utf-8') as f:
    f.write(dashboard_html)
print(f'HTML: {html_path} ({Path(html_path).stat().st_size/1e3:.0f} KB)')

meta = {
    'generated_at' : datetime.now().isoformat(),
    'source'       : 'GDELT v2 prototype',
    'country'      : 'Benin (BN)',
    'period_start' : str(df['date'].min().date()),
    'period_end'   : str(df['date'].max().date()),
    'rows'         : len(df),
    'columns'      : df.shape[1],
    'avg_tone'     : round(float(df['AvgTone'].mean()), 4),
    'goldstein'    : round(float(df['GoldsteinScale'].mean()), 4),
    'conflict_pct' : round(float(df['IsConflict'].mean()*100), 2),
    'features'     : list(df.columns),
}
meta_path = 'data/processed/dataset_metadata.json'
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
print(f'Meta: {meta_path}')

try:
    from google.colab import files
    print('\nTelechargement...')
    files.download(csv_path)
    files.download(meta_path)
    files.download(html_path)
    print('3 fichiers telecharges')
except ImportError:
    print('\nFichiers sauvegardes localement.')
    print('Ouvrir dashboard/index.html dans le navigateur')